<a href="https://colab.research.google.com/github/Kazi-Sadman/Drug-solubility-prediction/blob/main/Thesis_new_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ GPU CONNECTED")
    print("GPU name :", torch.cuda.get_device_name(0))
    print("Memory   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("❌ GPU NOT CONNECTED — CPU চলছে")

✅ GPU CONNECTED
GPU name : Tesla T4
Memory   : 15.64 GB


# =============================================================================
# THESIS: An Explainable Language-Aware Cross-Lingual Framework for
#         Bangla-English Fake News Detection Using Multilingual Transformers
#         and Classical Machine Learning
#
# FILE: Single-file end-to-end implementation for Google Colab / Jupyter
# SEED: 42  |  TEST SET: frozen until Section 26 (Final Evaluation)
# ===================================================


# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================
# Installs all dependencies needed for the full pipeline.
# Safe to re-run (pip skips already-installed packages).

In [1]:

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn \
    transformers datasets accelerate sentencepiece langdetect lime \
    joblib tqdm pyyaml openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 30.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# SECTION 2 — IMPORTS
# =============================================================================
# Import every library used later. Names are grouped by purpose.

# --- Standard library ---

In [2]:
# =============================================================================
# THESIS: An Explainable Language-Aware Cross-Lingual Framework for
#         Bangla-English Fake News Detection Using Multilingual Transformers
#         and Classical Machine Learning
#
# FILE: Single-file end-to-end implementation for Google Colab / Jupyter
# SEED: 42  |  TEST SET: frozen until Section 26 (Final Evaluation)
# =============================================================================


# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================
# Installs all dependencies needed for the full pipeline.
# Safe to re-run (pip skips already-installed packages).

!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn \
    transformers datasets accelerate sentencepiece langdetect lime \
    joblib tqdm pyyaml openpyxl


# =============================================================================
# SECTION 2 — IMPORTS
# =============================================================================
# Import every library used later. Names are grouped by purpose.

# --- Standard library ---
import os
import re
import sys
import json
import time
import html
import random
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Classical machine learning ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_fscore_support
)
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier

# --- Language identification ---
from langdetect import detect as detect_language_langdetect
from langdetect import DetectorFactory, LangDetectException
DetectorFactory.seed = 0

# --- Explainability ---
from lime.lime_text import LimeTextExplainer

# --- Deep learning (transformers) ---
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- Persistence ---
import joblib

# --- Notebook friendliness ---
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [3]:
# =============================================================================
# SECTION 3 — CONFIGURATION AND RANDOM SEEDS (Reproducibility)
# =============================================================================

# -------- 3.1 Random seed (reproducibility) --------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -------- 3.2 Dataset schema (from dataset audit) --------
COLUMN_NEWS_ID       = "id"
COLUMN_TITLE         = "title"
COLUMN_NEWS_BODY     = "news_details"
COLUMN_LANGUAGE      = "language"
COLUMN_CATEGORY      = "category"
COLUMN_LABEL         = "label"

LABEL_MAPPING        = {"Real": 0, "Fake": 1}
LABEL_NAMES          = ["Real", "Fake"]
NUMBER_OF_CLASSES    = 2

# Columns never used as model features (leakage / metadata risk)
METADATA_COLUMNS     = [COLUMN_NEWS_ID, COLUMN_CATEGORY]

# -------- 3.3 Output directory layout --------
OUTPUT_ROOT_DIR      = "outputs"
FIGURES_DIR          = f"{OUTPUT_ROOT_DIR}/figures"
REPORTS_DIR          = f"{OUTPUT_ROOT_DIR}/reports"
MODELS_DIR           = f"{OUTPUT_ROOT_DIR}/models"
PREDICTIONS_DIR      = f"{OUTPUT_ROOT_DIR}/predictions"
LIME_EXPLANATIONS_DIR= f"{OUTPUT_ROOT_DIR}/lime"

for output_directory in [
    OUTPUT_ROOT_DIR, FIGURES_DIR, REPORTS_DIR,
    MODELS_DIR, PREDICTIONS_DIR, LIME_EXPLANATIONS_DIR
]:
    Path(output_directory).mkdir(parents=True, exist_ok=True)

# -------- 3.4 Split ratios (Section 8) --------
TRAIN_FRACTION       = 0.70
VALIDATION_FRACTION  = 0.15
TEST_FRACTION        = 0.15

# -------- 3.5 TF-IDF hyperparameters (Section 11) --------
TFIDF_MIN_DOCUMENT_FREQUENCY = 2
TFIDF_MAX_DOCUMENT_FREQUENCY = 0.95
TFIDF_USE_SUBLINEAR_TF       = True
TFIDF_MAX_NUMBER_OF_FEATURES = 120_000

# N-gram configurations to evaluate during TF-IDF ablation
TFIDF_NGRAM_CANDIDATES = [(1, 1), (1, 2)]

# -------- 3.6 Classical model hyperparameter grids (Section 11) --------
SVM_REGULARIZATION_GRID       = [0.1, 1.0, 10.0]
LOGREG_REGULARIZATION_GRID    = [0.1, 1.0, 10.0]
NAIVE_BAYES_ALPHA_GRID        = [0.1, 0.5, 1.0]

XGBOOST_FIXED_PARAMETERS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": -1,
    "use_label_encoder": False,
}

# -------- 3.7 Cross-validation (Section 9) --------
NUMBER_OF_CROSS_VALIDATION_FOLDS = 5

# -------- 3.8 Transformer training (Section 14) --------
MULTILINGUAL_TRANSFORMER_NAMES = [
    "bert-base-multilingual-cased",   # mBERT
    "xlm-roberta-base",               # XLM-R
]
TRANSFORMER_MAX_SEQUENCE_LENGTH = 256
TRANSFORMER_BATCH_SIZE          = 16
TRANSFORMER_NUMBER_OF_EPOCHS    = 3
TRANSFORMER_LEARNING_RATE       = 2e-5
TRANSFORMER_WEIGHT_DECAY        = 0.01
TRANSFORMER_WARMUP_RATIO        = 0.1
TRANSFORMER_EARLY_STOP_PATIENCE = 1

# -------- 3.9 Leakage detection (Section 5 / 9) --------
NEAR_DUPLICATE_COSINE_THRESHOLD = 0.90

# -------- 3.10 LIME (Section 25) --------
LIME_NUMBER_OF_FEATURES = 15
LIME_NUMBER_OF_SAMPLES  = 1000

# -------- 3.11 Device --------
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Random seed set to {RANDOM_SEED}")
print(f"[INFO] Compute device: {COMPUTE_DEVICE}")
if COMPUTE_DEVICE == "cpu":
    print("[WARN] No GPU detected. Transformer fine-tuning will be SKIPPED.")


# =============================================================================
# Utility functions used throughout the notebook
# =============================================================================

def write_json_report(report_object, output_file_path):
    """
    Persist a Python object (dict/list) as a JSON file using UTF-8.
    Ensures Bangla characters are preserved.
    """
    Path(output_file_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, "w", encoding="utf-8") as json_file:
        json.dump(report_object, json_file, ensure_ascii=False,
                  indent=2, default=str)


def print_section_banner(section_number, section_title):
    """Print a clear section banner for readability in the notebook."""
    banner = "=" * 78
    print(f"\n{banner}\nSECTION {section_number} — {section_title}\n{banner}")



[INFO] Random seed set to 42
[INFO] Compute device: cuda


In [4]:
# =============================================================================
# SECTION 4 — DATASET LOADING
# =============================================================================
print_section_banner(4, "DATASET LOADING")

# In Google Colab, this cell prompts you to upload the CSV file.
# In Jupyter, place the CSV in the same folder as the notebook.
try:
    from google.colab import files as colab_files
    print("Please upload your dataset CSV (e.g., ben_fnd_16k (1).csv):")
    uploaded_files = colab_files.upload()
    RAW_DATASET_FILEPATH = list(uploaded_files.keys())[0]
    print(f"[INFO] Uploaded file: {RAW_DATASET_FILEPATH}")
except ImportError:
    RAW_DATASET_FILEPATH = "ben_fnd_16k (1).csv"
    print(f"[INFO] Not in Colab. Expecting local file: {RAW_DATASET_FILEPATH}")

# Fallback: try common alternative filenames
if not Path(RAW_DATASET_FILEPATH).exists():
    for alternative_filename in ["ben_fnd_16k.csv", "data.csv", "dataset.csv"]:
        if Path(alternative_filename).exists():
            RAW_DATASET_FILEPATH = alternative_filename
            break

assert Path(RAW_DATASET_FILEPATH).exists(), (
    f"Dataset not found at {RAW_DATASET_FILEPATH}. "
    "Please upload it or set RAW_DATASET_FILEPATH manually."
)

# Load with Python engine to tolerate malformed rows (audit detected one).
raw_dataset_dataframe = pd.read_csv(
    RAW_DATASET_FILEPATH,
    engine="python",
    on_bad_lines="skip",
)
raw_dataset_dataframe.columns = [column.strip() for column in raw_dataset_dataframe.columns]

print(f"[INFO] Raw dataset shape: {raw_dataset_dataframe.shape}")
print(f"[INFO] Columns: {list(raw_dataset_dataframe.columns)}")
raw_dataset_dataframe.head(3)




SECTION 4 — DATASET LOADING
Please upload your dataset CSV (e.g., ben_fnd_16k (1).csv):


Saving ben_fnd_16k (1).csv to ben_fnd_16k (1).csv
[INFO] Uploaded file: ben_fnd_16k (1).csv
[INFO] Raw dataset shape: (16349, 6)
[INFO] Columns: ['id', 'title', 'news_details', 'language', 'category', 'label']


,id,title,news_details,language,category,label
0,1,China’s GAC expands to Bangladesh,The showroom was launched in attendance of DHS...,English,Business,Real
1,2,Some Walmart garment orders from Bangladesh on...,Suppliers to Walmart have delayed nearly 1mill...,English,Business,Real
2,3,"After US tariffs, jobs hang by a thread in Ban...",A 35% U.S. tariff risks job losses in factorie...,English,Business,Real


In [5]:
# =============================================================================
# SECTION 5 — DATASET AUDIT (before any preprocessing or training)
# =============================================================================
print_section_banner(5, "DATASET AUDIT")

def audit_dataset_structure(dataframe, title_column, body_column,
                            label_column, language_column):
    """
    Produce a comprehensive dataset-audit dictionary.
    Reports shape, columns, missing values, duplicates, label / language
    distributions, text-length statistics, code-mixing and suspicious columns.
    """
    audit_result = {}

    # --- 5.1 Shape and column names ---
    audit_result["number_of_rows"]    = int(len(dataframe))
    audit_result["number_of_columns"] = int(dataframe.shape[1])
    audit_result["column_names"]      = list(dataframe.columns)

    # --- 5.2 Missing values per column ---
    audit_result["missing_values_per_column"] = {
        column_name: int(dataframe[column_name].isna().sum())
        for column_name in dataframe.columns
    }

    # --- 5.3 Label and language distribution ---
    audit_result["label_distribution"] = (
        dataframe[label_column].value_counts(dropna=False).to_dict()
    )
    audit_result["language_distribution"] = (
        dataframe[language_column].value_counts(dropna=False).to_dict()
    )
    audit_result["label_by_language"] = (
        dataframe.groupby([language_column, label_column])
                 .size()
                 .unstack(fill_value=0)
                 .to_dict()
    )

    # --- 5.4 Text-length statistics ---
    title_lengths = dataframe[title_column].fillna("").astype(str).str.len()
    body_lengths  = dataframe[body_column].fillna("").astype(str).str.len()
    combined_lengths = title_lengths + body_lengths

    audit_result["text_length_statistics"] = {
        "minimum_length":  int(combined_lengths.min()),
        "maximum_length":  int(combined_lengths.max()),
        "mean_length":     float(combined_lengths.mean()),
        "median_length":   float(combined_lengths.median()),
        "std_deviation":   float(combined_lengths.std()),
    }
    audit_result["number_of_empty_texts"]       = int((combined_lengths == 0).sum())
    audit_result["number_of_very_short_texts"]  = int((combined_lengths < 20).sum())

    # --- 5.5 Exact duplicates ---
    text_identity_key = (
        dataframe[title_column].fillna("").astype(str)
        + "||"
        + dataframe[body_column].fillna("").astype(str)
    )
    audit_result["number_of_exact_duplicate_rows"] = int(text_identity_key.duplicated().sum())
    audit_result["number_of_duplicate_titles"] = int(
        dataframe[title_column].fillna("").astype(str).str.strip().duplicated().sum()
    )
    audit_result["top_10_duplicated_bodies"] = (
        dataframe[body_column].fillna("").astype(str).str.strip()
                 .value_counts().head(10).to_dict()
    )

    # --- 5.6 Code-mixing detection (script overlap heuristic) ---
    bangla_script_pattern = re.compile(r"[\u0980-\u09FF]")
    latin_script_pattern  = re.compile(r"[A-Za-z]")

    bangla_rows = dataframe[language_column].astype(str).str.lower() == "bangla"
    english_rows = dataframe[language_column].astype(str).str.lower() == "english"

    audit_result["bangla_rows_containing_latin"] = int(
        (bangla_rows & dataframe[body_column].apply(
            lambda text: bool(latin_script_pattern.search(str(text)))
        )).sum()
    )
    audit_result["english_rows_containing_bangla"] = int(
        (english_rows & dataframe[body_column].apply(
            lambda text: bool(bangla_script_pattern.search(str(text)))
        )).sum()
    )

    # --- 5.7 Suspicious columns (potential leakage) ---
    suspicious_columns = []
    for column_name in dataframe.columns:
        if column_name in (title_column, body_column):
            continue
        try:
            unique_value_count = dataframe[column_name].nunique(dropna=True)
            if (unique_value_count > 0.9 * len(dataframe)
                    and column_name.lower() != "id"):
                suspicious_columns.append({
                    "column_name": column_name,
                    "reason": "very_high_cardinality",
                    "number_of_unique_values": int(unique_value_count),
                })
        except TypeError:
            pass
    audit_result["suspicious_columns"] = suspicious_columns

    return audit_result


dataset_audit_report = audit_dataset_structure(
    dataframe       = raw_dataset_dataframe,
    title_column    = COLUMN_TITLE,
    body_column     = COLUMN_NEWS_BODY,
    label_column    = COLUMN_LABEL,
    language_column = COLUMN_LANGUAGE,
)

write_json_report(dataset_audit_report,
                  f"{REPORTS_DIR}/dataset_audit_report.json")

print(f"[INFO] Dataset audit saved to {REPORTS_DIR}/dataset_audit_report.json")
print(json.dumps(
    {k: v for k, v in dataset_audit_report.items()
     if k != "top_10_duplicated_bodies"},
    indent=2, default=str,
)[:2500])


SECTION 5 — DATASET AUDIT
[INFO] Dataset audit saved to outputs/reports/dataset_audit_report.json
{
  "number_of_rows": 16349,
  "number_of_columns": 6,
  "column_names": [
    "id",
    "title",
    "news_details",
    "language",
    "category",
    "label"
  ],
  "missing_values_per_column": {
    "id": 0,
    "title": 1,
    "news_details": 1,
    "language": 1,
    "category": 0,
    "label": 0
  },
  "label_distribution": {
    "Real": 9441,
    "Fake": 6908
  },
  "language_distribution": {
    "English": 8883,
    "Bangla": 7465,
    "NaN": 1
  },
  "label_by_language": {
    "Fake": {
      "Bangla": 2684,
      "English": 4223
    },
    "Real": {
      "Bangla": 4781,
      "English": 4660
    }
  },
  "text_length_statistics": {
    "minimum_length": 18,
    "maximum_length": 26791,
    "mean_length": 1988.7000428160743,
    "median_length": 1726.0,
    "std_deviation": 1486.2608529173926
  },
  "number_of_empty_texts": 0,
  "number_of_very_short_texts": 1,
  "number_of_ex

In [6]:
# =============================================================================
# SECTION 6 — PREPROCESSING
# =============================================================================
print_section_banner(6, "PREPROCESSING")

# Precompiled regex patterns (module-level for speed)
URL_PATTERN             = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN           = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_PATTERN        = re.compile(r"<[^>]+>")
CONTROL_CHARACTER_PATTERN = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WHITESPACE_PATTERN      = re.compile(r"\s+")
REPEATED_CHARACTER_PATTERN = re.compile(r"(.)\1{3,}")
ZERO_WIDTH_PATTERN      = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]")

# Bangla digit → ASCII digit mapping
BANGLA_TO_ASCII_DIGIT_TRANSLATION = {
    ord(bangla_digit): ord("0") + index
    for index, bangla_digit in enumerate("০১২৩৪৫৬৭৮৯")
}


def normalize_text_for_modelling(raw_text):
    """
    Apply script-agnostic text normalization:
      - HTML entity decoding
      - Unicode NFC normalization (fixes Bangla combining forms)
      - HTML tag removal
      - URL and email removal
      - Bangla digit → ASCII digit
      - Zero-width character removal
      - Control character removal
      - Repeated character collapsing (3+ → 2)
      - Whitespace collapsing
    Preserves text case and Bangla characters.
    """
    if not isinstance(raw_text, str):
        return ""

    text = html.unescape(raw_text)
    text = unicodedata.normalize("NFC", text)
    text = HTML_TAG_PATTERN.sub(" ", text)
    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = text.translate(BANGLA_TO_ASCII_DIGIT_TRANSLATION)
    text = ZERO_WIDTH_PATTERN.sub("", text)
    text = CONTROL_CHARACTER_PATTERN.sub(" ", text)
    text = REPEATED_CHARACTER_PATTERN.sub(r"\1\1", text)
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    return text


def preprocess_dataset(dataframe, title_column, body_column,
                       label_column, language_column, label_mapping):
    """
    Apply text normalization, build the combined text representation
    'title [SEP] body', drop empty rows, encode labels, normalize language.
    Returns a cleaned DataFrame.
    """
    dataframe = dataframe.copy()

    dataframe[title_column] = dataframe[title_column].fillna("").map(normalize_text_for_modelling)
    dataframe[body_column]  = dataframe[body_column].fillna("").map(normalize_text_for_modelling)

    # Primary modelling representation: title + [SEP] + body
    dataframe["combined_text"] = (
        dataframe[title_column] + " [SEP] " + dataframe[body_column]
    ).map(normalize_text_for_modelling)

    # Drop rows with empty combined text
    rows_before_cleaning = len(dataframe)
    dataframe = dataframe[dataframe["combined_text"].str.strip().str.len() > 0]
    dataframe = dataframe.reset_index(drop=True)
    rows_after_cleaning = len(dataframe)
    print(f"[INFO] Removed {rows_before_cleaning - rows_after_cleaning} empty rows; "
          f"{rows_after_cleaning} rows remain.")

    # Encode labels
    dataframe["label_id"] = dataframe[label_column].map(label_mapping)
    dataframe = dataframe.dropna(subset=["label_id"]).reset_index(drop=True)
    dataframe["label_id"] = dataframe["label_id"].astype(int)

    # Normalize language string
    dataframe[language_column] = (
        dataframe[language_column].astype(str).str.strip().str.lower()
    )

    return dataframe


preprocessed_dataframe = preprocess_dataset(
    dataframe       = raw_dataset_dataframe,
    title_column    = COLUMN_TITLE,
    body_column     = COLUMN_NEWS_BODY,
    label_column    = COLUMN_LABEL,
    language_column = COLUMN_LANGUAGE,
    label_mapping   = LABEL_MAPPING,
)

print(f"[INFO] Preprocessed shape: {preprocessed_dataframe.shape}")
preprocessed_dataframe[["combined_text", "label_id", COLUMN_LANGUAGE]].head(3)


SECTION 6 — PREPROCESSING
[INFO] Removed 0 empty rows; 16349 rows remain.
[INFO] Preprocessed shape: (16349, 8)


,combined_text,label_id,language
0,China’s GAC expands to Bangladesh [SEP] The sh...,0,english
1,Some Walmart garment orders from Bangladesh on...,0,english
2,"After US tariffs, jobs hang by a thread in Ban...",0,english
